# Aplicação de empréstimo: sistema do parceiro usando a API de Score

Este notebook simula o sistema de uma **empresa parceira da QuantumFinance**: um pedido de empréstimo é
cadastrado, o sistema chama a API de score no Cloud Run e devolve **SIM** ou **NÃO**, com o motivo e a
versão do modelo que embasou a decisão.

| Etapa | O que acontece |
|---|---|
| 1. Cadastro | dados financeiros do cliente + valor e prazo pedidos |
| 2. Consulta | `POST /v1/score` (um pedido) ou `POST /v1/score/lote` (fila de pedidos) com o header `X-API-Key` |
| 3. Política de crédito | regra de negócio do parceiro aplicada sobre o score e as probabilidades |
| 4. Registro | cada decisão fica no histórico com `id_requisicao`, versão do modelo e versão da release (auditoria) |

**Chave da API:** peça a sua ao responsável pelo projeto. No Colab, clique na chave 🔑 (Secrets) à esquerda,
crie o segredo `QF_API_KEY`, cole a chave e ative o acesso do notebook. Se preferir, rode a célula de
configuração e cole a chave quando ela for pedida (o valor fica oculto).

In [ ]:
# Configuração
API_URL = "https://api-score-credito-czkhbhag2q-rj.a.run.app"
TESTE_URL = "https://raw.githubusercontent.com/rfc890305/mlop_score_credito/main/data/raw/test.csv"

import os, getpass, time, requests
from datetime import datetime, timezone
import numpy as np, pandas as pd

try:
    from google.colab import userdata
    API_KEY = userdata.get("QF_API_KEY")
except Exception:
    API_KEY = os.environ.get("QF_API_KEY") or getpass.getpass("Chave da API (X-API-Key): ")
API_KEY = (API_KEY or "").strip().strip("\"'")
HEADERS = {"X-API-Key": API_KEY}
print(f"Chave carregada: {len(API_KEY)} caracteres ({API_KEY[:4]}...{API_KEY[-4:]})")

r = requests.get(f"{API_URL}/v1/modelo", headers=HEADERS, timeout=60)
if r.status_code != 200:
    raise SystemExit(f"A API recusou a chamada: HTTP {r.status_code} {r.text}")
MODELO = r.json()
print(f"Modelo em produção: v{MODELO['versao']} ({MODELO['algoritmo']}), F1 macro na validação = "
      f"{MODELO.get('f1_macro_validacao')}, release {MODELO.get('versao_release')}")

## Política de crédito do parceiro

A API devolve o **score** (`Poor`, `Standard` ou `Good`) e a probabilidade de cada classe. A decisão de
conceder o empréstimo é do parceiro. Aqui, ela segue estas regras, na ordem:

1. Score **Poor** → **NÃO** (risco alto).
2. A parcela do novo empréstimo somada às parcelas atuais (`Total_EMI_per_month`) não pode passar de
   **35% da renda líquida mensal** → caso contrário, **NÃO**.
3. Score **Standard** só é aprovado se a probabilidade de `Poor` for menor que **30%**.
4. Nos demais casos → **SIM**.

A parcela usa a tabela Price com juros de 2,5% ao mês. Os parâmetros ficam em `POLITICA` e podem ser ajustados.

In [ ]:
POLITICA = {"juros_mensal": 0.025, "comprometimento_maximo": 0.35, "prob_poor_maxima_standard": 0.30}

def parcela_price(valor, prazo_meses, juros=POLITICA["juros_mensal"]):
    return valor * juros / (1 - (1 + juros) ** -prazo_meses)

def decidir(cliente, resultado, valor, prazo_meses):
    score, probs = resultado["score"], resultado["probabilidades"]
    parcela = parcela_price(valor, prazo_meses)
    comprometimento = (parcela + cliente["Total_EMI_per_month"]) / max(cliente["Monthly_Inhand_Salary"], 1)
    if score == "Poor":
        return "NÃO", f"score Poor (probabilidade de Poor {probs['Poor']:.0%})", parcela, comprometimento
    if comprometimento > POLITICA["comprometimento_maximo"]:
        return "NÃO", f"parcelas comprometeriam {comprometimento:.0%} da renda (máximo {POLITICA['comprometimento_maximo']:.0%})", parcela, comprometimento
    if score == "Standard" and probs["Poor"] >= POLITICA["prob_poor_maxima_standard"]:
        return "NÃO", f"score Standard com risco de Poor de {probs['Poor']:.0%}", parcela, comprometimento
    return "SIM", f"score {score} e parcela compatível com a renda ({comprometimento:.0%})", parcela, comprometimento

HISTORICO_ARQUIVO = "historico_decisoes.csv"
HISTORICO = pd.read_csv(HISTORICO_ARQUIVO, dtype={"versao_modelo": str}).to_dict("records") if os.path.exists(HISTORICO_ARQUIVO) else []
print(f"Histórico carregado: {len(HISTORICO)} decisões anteriores")

def registrar(pedido_id, cliente, valor, prazo, resultado, modelo, id_requisicao, origem):
    decisao, motivo, parcela, comp = decidir(cliente, resultado, valor, prazo)
    HISTORICO.append({
        "data": datetime.now(timezone.utc).isoformat(timespec="seconds"), "origem": origem,
        "pedido_id": pedido_id, "cliente_id": cliente.get("cliente_id"), "valor": valor, "prazo_meses": prazo,
        "parcela": round(parcela, 2), "comprometimento_renda": round(comp, 3),
        "score": resultado["score"], "prob_Poor": resultado["probabilidades"]["Poor"],
        "prob_Good": resultado["probabilidades"]["Good"], "decisao": decisao, "motivo": motivo,
        "versao_modelo": str(modelo["versao"]), "versao_release": modelo.get("versao_release"),
        "id_requisicao": id_requisicao,
    })
    pd.DataFrame(HISTORICO).to_csv(HISTORICO_ARQUIVO, index=False)
    return decisao, motivo, parcela, comp

## 1. Um pedido de empréstimo (formulário)

Escolha um cliente de exemplo (dados reais de setembro do `test.csv`), o valor e o prazo, e rode a célula.
Os clientes A a D foram escolhidos para mostrar os casos típicos. O **Cliente D** é um caso limítrofe: no
modelo com a configuração inicial ele tende a sair `Poor` (NÃO) e, com o modelo melhorado, `Standard` com
risco baixo (SIM). É um bom caso para comparar antes e depois de uma nova versão.

In [ ]:
CLIENTES_EXEMPLO = {
    "A": {
        "cliente_id": "CUS_0xaedb",
        "Age": 20,
        "Annual_Income": 85554.03,
        "Monthly_Inhand_Salary": 7185.5,
        "Num_Bank_Accounts": 4,
        "Num_Credit_Card": 2,
        "Interest_Rate": 3.0,
        "Num_of_Loan": 0,
        "Delay_from_due_date": 10,
        "Num_of_Delayed_Payment": 4,
        "Changed_Credit_Limit": 6.43,
        "Num_Credit_Inquiries": 3,
        "Outstanding_Debt": 1095.73,
        "Credit_Utilization_Ratio": 37.22,
        "Credit_History_Age": 243,
        "Total_EMI_per_month": 0.0,
        "Amount_invested_monthly": 212.01,
        "Monthly_Balance": 776.54,
        "Credit_Mix": "Good",
        "Payment_of_Min_Amount": "No",
        "Payment_Behaviour": "Low_spent_Large_value_payments"
    },
    "B": {
        "cliente_id": "CUS_0x3e45",
        "Age": 36,
        "Annual_Income": 54392.16,
        "Monthly_Inhand_Salary": 4766.68,
        "Num_Bank_Accounts": 6,
        "Num_Credit_Card": 4,
        "Interest_Rate": 14.0,
        "Num_of_Loan": 3,
        "Delay_from_due_date": 10,
        "Num_of_Delayed_Payment": 8,
        "Changed_Credit_Limit": 5.54,
        "Num_Credit_Inquiries": 7,
        "Outstanding_Debt": 179.22,
        "Credit_Utilization_Ratio": 33.5,
        "Credit_History_Age": 330,
        "Total_EMI_per_month": 124.39,
        "Amount_invested_monthly": 329.55,
        "Monthly_Balance": 292.72,
        "Credit_Mix": "Standard",
        "Payment_of_Min_Amount": "Yes",
        "Payment_Behaviour": "Low_spent_Large_value_payments"
    },
    "C": {
        "cliente_id": "CUS_0xb14",
        "Age": 26,
        "Annual_Income": 49967.01,
        "Monthly_Inhand_Salary": 4091.92,
        "Num_Bank_Accounts": 6,
        "Num_Credit_Card": 5,
        "Interest_Rate": 31.0,
        "Num_of_Loan": 6,
        "Delay_from_due_date": 27,
        "Num_of_Delayed_Payment": 13,
        "Changed_Credit_Limit": 19.87,
        "Num_Credit_Inquiries": 12,
        "Outstanding_Debt": 2253.95,
        "Credit_Utilization_Ratio": 29.13,
        "Credit_History_Age": 210,
        "Total_EMI_per_month": 156.0,
        "Amount_invested_monthly": 450.93,
        "Monthly_Balance": 92.26,
        "Credit_Mix": "Standard",
        "Payment_of_Min_Amount": "Yes",
        "Payment_Behaviour": "Low_spent_Small_value_payments"
    },
    "D": {
        "cliente_id": "CUS_0x77ac",
        "Age": 33,
        "Annual_Income": 39232.16,
        "Monthly_Inhand_Salary": 3331.35,
        "Num_Bank_Accounts": 5,
        "Num_Credit_Card": 7,
        "Interest_Rate": 1.0,
        "Num_of_Loan": 0,
        "Delay_from_due_date": 15,
        "Num_of_Delayed_Payment": 10,
        "Changed_Credit_Limit": 11.32,
        "Num_Credit_Inquiries": 5,
        "Outstanding_Debt": 487.44,
        "Credit_Utilization_Ratio": 29.47,
        "Credit_History_Age": 324,
        "Total_EMI_per_month": 0.0,
        "Amount_invested_monthly": 77.35,
        "Monthly_Balance": 505.78,
        "Credit_Mix": "Good",
        "Payment_of_Min_Amount": "NM",
        "Payment_Behaviour": "High_spent_Medium_value_payments"
    }
}

perfil = "D - caso limítrofe" #@param ["A - bom histórico", "B - histórico mediano", "C - alto risco", "D - caso limítrofe"]
valor_solicitado = 6000 #@param {type:"number"}
prazo_meses = 24 #@param [12, 24, 36, 48] {type:"raw"}

cliente = CLIENTES_EXEMPLO[perfil[0]]
resposta = requests.post(f"{API_URL}/v1/score", headers=HEADERS, json=cliente, timeout=60)
if resposta.status_code != 200:
    raise SystemExit(f"Erro da API: HTTP {resposta.status_code} {resposta.text}")
corpo = resposta.json()
decisao, motivo, parcela, comp = registrar(f"form-{perfil[0]}-{int(valor_solicitado)}-{prazo_meses}", cliente,
                                           valor_solicitado, prazo_meses, corpo, corpo["modelo"],
                                           corpo["id_requisicao"], "formulário")
print("=" * 64)
print(f" PEDIDO DE EMPRÉSTIMO: cliente {cliente['cliente_id']}")
print(f" Valor: {valor_solicitado:,.2f} em {prazo_meses}x de {parcela:,.2f}")
print(f" Score: {corpo['score']}  |  " + "  ".join(f"{k}: {v:.1%}" for k, v in corpo["probabilidades"].items()))
print(f" DECISÃO: {decisao}  ({motivo})")
print(f" Modelo v{corpo['modelo']['versao']} | release {corpo['modelo'].get('versao_release')} | requisição {corpo['id_requisicao']}")
print("=" * 64)

## 2. Fila de pedidos do dia (lote)

Simula 30 pedidos de clientes de setembro (`test.csv`, dados que o modelo nunca viu), com valor e prazo
sorteados com semente fixa, para que a mesma fila possa ser reprocessada depois de uma nova versão.
Os 30 pedidos vão numa única chamada a `/v1/score/lote`.

In [ ]:
FAIXAS = {
    "Age": (14, 100), "Annual_Income": (0, 250_000), "Monthly_Inhand_Salary": (0, 25_000),
    "Num_Bank_Accounts": (0, 20), "Num_Credit_Card": (0, 15), "Interest_Rate": (0, 40),
    "Num_of_Loan": (0, 15), "Delay_from_due_date": (-10, 100), "Num_of_Delayed_Payment": (0, 40),
    "Changed_Credit_Limit": (-50, 50), "Num_Credit_Inquiries": (0, 30), "Outstanding_Debt": (0, 10_000),
    "Credit_Utilization_Ratio": (0, 100), "Credit_History_Age": (0, 600), "Total_EMI_per_month": (0, 5_000),
    "Amount_invested_monthly": (0, 9_999), "Monthly_Balance": (-5_000, 5_000),
}
INTEIROS = ["Age", "Num_Bank_Accounts", "Num_Credit_Card", "Num_of_Loan", "Delay_from_due_date",
            "Num_of_Delayed_Payment", "Num_Credit_Inquiries", "Credit_History_Age"]
CATEGORIAS = {
    "Credit_Mix": ["Bad", "Standard", "Good"],
    "Payment_of_Min_Amount": ["Yes", "No", "NM"],
    "Payment_Behaviour": [f"{g}_spent_{v}_value_payments" for g in ("Low", "High") for v in ("Small", "Medium", "Large")],
}

def numero(s):
    return pd.to_numeric(s.astype("string").str.strip().str.strip("_"), errors="coerce").astype("float64")

def historico_meses(s):
    x = s.astype("string").str.extract(r"(\d+)\s*Years?\s*and\s*(\d+)\s*Months?")
    return x[0].astype("float64") * 12 + x[1].astype("float64")

bruto = pd.read_csv(TESTE_URL, low_memory=False)
setembro = bruto[bruto["Month"] == "September"]
dados = pd.DataFrame({"cliente_id": setembro["Customer_ID"].astype(str)}, index=setembro.index)
for col, (mn, mx) in FAIXAS.items():
    v = historico_meses(setembro[col]) if col == "Credit_History_Age" else numero(setembro[col])
    dados[col] = v.where(v.between(mn, mx))
for col, validos in CATEGORIAS.items():
    dados[col] = setembro[col].where(setembro[col].isin(validos))

rng = np.random.default_rng(7)
fila = dados.dropna().sample(30, random_state=7)
fila["valor"] = (fila["Annual_Income"] * rng.uniform(0.05, 0.40, len(fila))).round(-2)
fila["prazo"] = rng.choice([12, 24, 36, 48], len(fila))
clientes = []
for linha in fila.drop(columns=["valor", "prazo"]).to_dict("records"):
    for c in INTEIROS:
        linha[c] = int(linha[c])
    clientes.append(linha)

r = requests.post(f"{API_URL}/v1/score/lote", headers=HEADERS, json={"clientes": clientes}, timeout=120)
if r.status_code != 200:
    raise SystemExit(f"Erro da API: HTTP {r.status_code} {r.text}")
lote = r.json()
for cliente, (_, pedido), resultado in zip(clientes, fila.iterrows(), lote["resultados"]):
    registrar(f"fila-{cliente['cliente_id']}", cliente, float(pedido["valor"]), int(pedido["prazo"]),
              resultado, lote["modelo"], lote["id_requisicao"], "fila do dia")

hoje = pd.DataFrame(HISTORICO[-len(clientes):])
print(f"Modelo v{lote['modelo']['versao']} (release {lote['modelo'].get('versao_release')}): "
      f"{(hoje['decisao'] == 'SIM').mean():.0%} dos pedidos aprovados")
print(pd.crosstab(hoje["score"], hoje["decisao"], margins=True, margins_name="Total"))
hoje[["cliente_id", "valor", "prazo_meses", "parcela", "comprometimento_renda", "score", "prob_Poor", "decisao", "motivo"]]

## 3. Histórico e comparação entre versões do modelo

Todas as decisões ficam em `historico_decisoes.csv`, com a versão do modelo que as embasou.

**Para comparar versões:** rode as seções 1 e 2, espere o deploy de uma nova versão (ou faça um rollback)
e rode as duas seções de novo, na mesma sessão. Se a sessão do Colab for reiniciada, envie antes o
`historico_decisoes.csv` baixado pelo painel de arquivos: ele é recarregado automaticamente. A tabela
abaixo mostra, para cada pedido, a decisão de cada versão e destaca os que mudaram.

In [ ]:
hist = pd.DataFrame(HISTORICO)
print("Decisões por versão do modelo:")
print(pd.crosstab([hist["versao_modelo"], hist["versao_release"]], hist["decisao"]))

ultimas = hist.drop_duplicates(["pedido_id", "versao_modelo"], keep="last")
comparacao = ultimas.pivot(index="pedido_id", columns="versao_modelo", values="decisao")
if comparacao.shape[1] < 2:
    print("\nAinda há só uma versão no histórico. Rode as seções 1 e 2 de novo depois de uma nova versão.")
else:
    comparacao.columns = [f"v{c}" for c in comparacao.columns]
    mudaram = comparacao[comparacao.nunique(axis=1) > 1]
    print(f"\n{len(mudaram)} de {len(comparacao)} pedidos mudaram de decisão entre as versões:")
    detalhe = ultimas[ultimas["pedido_id"].isin(mudaram.index)].pivot(
        index="pedido_id", columns="versao_modelo", values=["score", "prob_Poor", "decisao"])
    display(detalhe)

In [ ]:
# Baixar o histórico (auditoria das decisões)
try:
    from google.colab import files
    files.download(HISTORICO_ARQUIVO)
except Exception:
    print(f"Histórico salvo em {os.path.abspath(HISTORICO_ARQUIVO)}")